# CNN Image Classification

This notebook trains a Convolutional Neural Network (CNN) to classify images from various datasets, then evaluates performance with metrics and visualizations.

**Supported datasets:**
- **CIFAR-10**: 32x32 color images in 10 classes (airplanes, cars, birds, etc.)
- **MNIST**: 28x28 grayscale handwritten digits (0-9)
- **Fashion-MNIST**: 28x28 grayscale clothing items (t-shirts, trousers, shoes, etc.)

The model architecture automatically adapts to the input dimensions and number of channels for each dataset.

<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/image-classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Now we'll import the necessary libraries and enable autoreload so changes to our shared library are automatically loaded.

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as L
import wandb
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Import shared utilities from local package
from aiml_notebooks import (
    log_gradients, log_model_weights, log_gradient_flow,
    create_trainer, create_dataset, create_dataloaders,
    CIFAR10_CLASSES, CIFAR10_MEAN, CIFAR10_STD,
    MNIST_CLASSES, MNIST_MEAN, MNIST_STD,
    FASHIONMNIST_CLASSES, FASHIONMNIST_MEAN, FASHIONMNIST_STD,
)

# Enable autoreload for hot reloading of library changes
%load_ext autoreload
%autoreload 2

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {L.__version__}")

In [ ]:
# Configuration (Base defaults - can be overridden by papermill parameters)
CONFIG = {
    # Data
    'dataset': 'cifar10',            # Dataset to use: 'cifar10', 'mnist', or 'fashionmnist'
    'seed': 42,                      # Random seed for reproducibility
    'batch_size': 128,               # Number of examples per training batch
    'num_workers': 4,                # Number of parallel data loading workers
    
    # Model
    'num_conv_layers': 3,            # Number of convolutional blocks
    'base_channels': 32,             # Number of channels in first conv layer (doubles each block)
    'dropout': 0.5,                  # Dropout rate to prevent overfitting
    'learning_rate': 1e-3,           # Step size for optimizer (0.001)
    
    # Training
    'max_epochs': 50,                # Number of complete passes through training data
    'log_every_n_steps': 20,         # How often to log training metrics
    
    # Weights & Biases
    'wandb_project': 'cnn-image-classification',  # W&B project name
    'wandb_run_name': None,          # Optional run name (None = auto-generated)
}

# Set random seeds
L.seed_everything(CONFIG['seed'])

These papermill parameters allow hyperparameter sweeps to override the default CONFIG values.

Now we'll use the dataset factory to load the selected dataset:
- **CIFAR-10**: 60,000 32x32 color images in 10 classes (50k train, 10k test)
- **MNIST**: 70,000 28x28 grayscale handwritten digits (60k train, 10k test)
- **Fashion-MNIST**: 70,000 28x28 grayscale clothing images in 10 classes (60k train, 10k test)

In [ ]:
# Dataset configuration mapping
DATASET_CONFIG = {
    'cifar10': {
        'classes': CIFAR10_CLASSES,
        'mean': CIFAR10_MEAN,
        'std': CIFAR10_STD,
        'num_channels': 3,
        'image_size': 32,
        'name': 'CIFAR-10',
    },
    'mnist': {
        'classes': MNIST_CLASSES,
        'mean': MNIST_MEAN,
        'std': MNIST_STD,
        'num_channels': 1,
        'image_size': 28,
        'name': 'MNIST',
    },
    'fashionmnist': {
        'classes': FASHIONMNIST_CLASSES,
        'mean': FASHIONMNIST_MEAN,
        'std': FASHIONMNIST_STD,
        'num_channels': 1,
        'image_size': 28,
        'name': 'Fashion-MNIST',
    },
}

# Get dataset configuration
dataset_config = DATASET_CONFIG[CONFIG['dataset']]
class_names = dataset_config['classes']
num_classes = len(class_names)
num_channels = dataset_config['num_channels']
image_size = dataset_config['image_size']

# Define data transforms (normalization and data augmentation for training)
# Training transforms with data augmentation
if CONFIG['dataset'] == 'cifar10':
    # CIFAR-10 specific augmentation
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(image_size, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(dataset_config['mean'], dataset_config['std'])
    ])
elif CONFIG['dataset'] in ['mnist', 'fashionmnist']:
    # MNIST/Fashion-MNIST specific augmentation
    train_transform = transforms.Compose([
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(dataset_config['mean'], dataset_config['std'])
    ])

# Test transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(dataset_config['mean'], dataset_config['std'])
])

# Load dataset using the factory (returns train, test)
train_dataset, test_dataset = create_dataset(
    dataset_id=CONFIG['dataset'],
    train_transform=train_transform,
    test_transform=test_transform
)

print(f"Dataset: {dataset_config['name']}")
print(f"Image size: {image_size}x{image_size}x{num_channels}")
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes ({num_classes}): {', '.join(class_names)}")

Now we'll use the dataloader factory to create batched, shuffled loaders for training and testing.

In [ ]:
# Create data loaders using the factory
train_loader, test_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=test_dataset,  # Using test set as val set for this notebook
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    use_collate_fn=False,  # Vision datasets don't need padding
    pin_memory=True
)

# Display sample images
print("\nSample Images:")
print("="*50)

# Get a batch of training data
sample_images, sample_labels = next(iter(train_loader))

# Plot first 8 images
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    # Denormalize image for display
    img = sample_images[i].numpy()
    
    if num_channels == 1:
        # Grayscale image (MNIST, Fashion-MNIST)
        img = img[0]  # Remove channel dimension
        img = img * dataset_config['std'][0] + dataset_config['mean'][0]
        img = np.clip(img, 0, 1)
        ax.imshow(img, cmap='gray')
    else:
        # Color image (CIFAR-10)
        img = img.transpose(1, 2, 0)  # CHW -> HWC
        img = img * np.array(dataset_config['std']) + np.array(dataset_config['mean'])
        img = np.clip(img, 0, 1)
        ax.imshow(img)
    
    ax.set_title(class_names[sample_labels[i]])
    ax.axis('off')

plt.tight_layout()
plt.show()
print("="*50)

Now we'll define our CNN model with convolutional blocks, batch normalization, dropout, and a classifier.

In [ ]:
# Define the CNN model with convolutional blocks and fully connected classifier
class ImageClassifierCNN(L.LightningModule):
    def __init__(
        self, 
        num_classes: int = 10,
        in_channels: int = 3,
        input_size: int = 32,
        num_conv_layers: int = 3,
        base_channels: int = 32,
        dropout: float = 0.5,
        learning_rate: float = 1e-3,
    ):
        super().__init__()
        
        self.save_hyperparameters()
        
        # Build convolutional layers dynamically
        conv_layers = []
        channels = in_channels
        
        for i in range(num_conv_layers):
            out_channels = base_channels * (2 ** i)  # Double channels each layer
            
            # Convolutional block: Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm -> ReLU -> MaxPool
            conv_layers.extend([
                nn.Conv2d(channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2, stride=2)  # Halves spatial dimensions
            ])
            
            channels = out_channels
        
        self.conv_layers = nn.Sequential(*conv_layers)
        
        # Calculate the size of the flattened features
        # Input size after N pooling layers: input_size / (2^N)
        final_spatial_size = input_size // (2 ** num_conv_layers)
        final_channels = base_channels * (2 ** (num_conv_layers - 1))
        flattened_size = final_channels * final_spatial_size * final_spatial_size
        
        # Fully connected classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
        
        # Cross-entropy loss function
        self.criterion = nn.CrossEntropyLoss()
        
        # Track predictions for evaluation
        self.test_predictions = []
        self.test_labels = []
    
    def forward(self, x):
        # Pass through convolutional layers
        features = self.conv_layers(x)
        
        # Pass through classifier
        logits = self.classifier(features)
        
        return logits
    
    def training_step(self, batch, batch_idx):
        # Forward pass to get logits for each image in batch
        x, y = batch
        logits = self(x)
        
        # Compute the loss between the logits and the true labels
        loss = self.criterion(logits, y)
        
        # Calculate accuracy
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        
        # Log the loss and accuracy for this batch
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        
        # Log gradients every N steps (reduce overhead)
        if batch_idx % 10 == 0:  # TODO: softcode logging frequency
            log_gradients(self, step=self.global_step)
        
        # Return the loss for this batch
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        
        # Forward pass to get logits for each image in batch
        logits = self(x)
        
        # Compute the loss between the logits and the true labels
        loss = self.criterion(logits, y)
        
        # Calculate accuracy
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        
        # Log the validation loss and accuracy for this batch
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        
        return loss
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        
        # Forward pass to get logits for each image in batch
        logits = self(x)
        
        # Calculate predictions
        preds = torch.argmax(logits, dim=1)
        
        # Store predictions and labels for confusion matrix
        self.test_predictions.extend(preds.cpu().numpy())
        self.test_labels.extend(y.cpu().numpy())
        
        # Calculate accuracy
        acc = (preds == y).float().mean()
        self.log('test_acc', acc, prog_bar=True)
        
        return acc
    
    def on_train_epoch_end(self):
        # Log detailed gradient flow visualization at end of each epoch
        log_gradient_flow(self, step=self.global_step)
        log_model_weights(self, step=self.global_step)
    
    def configure_optimizers(self):
        # TODO: softcode optimizer
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        
        # Learning rate scheduler (reduce on plateau)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode='min', 
            factor=0.5, 
            patience=5,
            verbose=True
        )
        
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss'
            }
        }

# Initialize the model with CONFIG hyperparameters and dataset-specific parameters
model = ImageClassifierCNN(
    num_classes=num_classes,
    in_channels=num_channels,
    input_size=image_size,
    num_conv_layers=CONFIG['num_conv_layers'],
    base_channels=CONFIG['base_channels'],
    dropout=CONFIG['dropout'],
    learning_rate=CONFIG['learning_rate'],
)

print(f"Initialized CNN for {dataset_config['name']}")
print(f"Input: {num_channels} x {image_size} x {image_size}")
print(f"Output: {num_classes} classes")
print(f"Architecture: {CONFIG['num_conv_layers']} convolutional blocks")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Now we'll train the model using PyTorch Lightning's Trainer with W&B logging.

In [ ]:
# Train the model (W&B logger created automatically)
trainer = create_trainer(
    max_epochs=CONFIG['max_epochs'],
    log_every_n_steps=CONFIG['log_every_n_steps'],
    wandb_project=CONFIG['wandb_project'],
    wandb_run_name=CONFIG['wandb_run_name'],
    wandb_config=CONFIG,
    model=model
)
trainer.fit(model, train_loader, test_loader)

Now we'll evaluate the model on the test set and visualize the results.

In [ ]:
# Test the model
print("\n" + "="*50)
print("TESTING ON TEST SET")
print("="*50)

# Reset predictions and labels
model.test_predictions = []
model.test_labels = []

# Run test
test_results = trainer.test(model, test_loader)

# Calculate and display test accuracy
test_acc = test_results[0]['test_acc']
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

# Log to W&B
wandb.log({
    'test_accuracy': test_acc * 100
})

Now we'll create a confusion matrix to see which classes are most commonly confused.

In [ ]:
# Create confusion matrix
cm = confusion_matrix(model.test_labels, model.test_predictions)

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Calculate per-class accuracy
print("\nPer-Class Accuracy:")
print("="*50)
for i, class_name in enumerate(class_names):
    class_acc = cm[i, i] / cm[i].sum() * 100
    print(f"{class_name:12s}: {class_acc:5.2f}%")
print("="*50)

# Log confusion matrix to W&B
wandb.log({
    'confusion_matrix': wandb.Image(plt.gcf())
})

Finally, let's visualize some sample predictions to see how the model performs on individual images.

In [ ]:
# Display sample predictions
@torch.no_grad()
def show_predictions(model, data_loader, num_samples=16):
    model.eval()
    
    # Get a batch
    images, labels = next(iter(data_loader))
    images = images[:num_samples]
    labels = labels[:num_samples]
    
    # Get predictions
    logits = model(images.to(model.device))
    preds = torch.argmax(logits, dim=1).cpu()
    probs = F.softmax(logits, dim=1).cpu()
    confidences = probs.max(dim=1)[0]
    
    # Plot
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    for i, ax in enumerate(axes.flat):
        if i >= len(images):
            break
            
        # Denormalize image for display
        img = images[i].numpy()
        
        if num_channels == 1:
            # Grayscale image (MNIST, Fashion-MNIST)
            img = img[0]  # Remove channel dimension
            img = img * dataset_config['std'][0] + dataset_config['mean'][0]
            img = np.clip(img, 0, 1)
            ax.imshow(img, cmap='gray')
        else:
            # Color image (CIFAR-10)
            img = img.transpose(1, 2, 0)  # CHW -> HWC
            img = img * np.array(dataset_config['std']) + np.array(dataset_config['mean'])
            img = np.clip(img, 0, 1)
            ax.imshow(img)
        
        # Color code: green if correct, red if incorrect
        correct = preds[i] == labels[i]
        color = 'green' if correct else 'red'
        
        # Title with prediction and confidence
        title = f"True: {class_names[labels[i]]}\n"
        title += f"Pred: {class_names[preds[i]]} ({confidences[i]*100:.1f}%)"
        ax.set_title(title, color=color, fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

print("\nSample Predictions:")
print("="*50)
print("Green = Correct, Red = Incorrect")
print("="*50)
show_predictions(model, test_loader, num_samples=16)

# Log sample predictions to W&B
wandb.log({
    'sample_predictions': wandb.Image(plt.gcf())
})